# Model Evaluation and Alarm Prediction Visualization
### Honeywell Temperature Alarm Model (`03TIC_1023.PV` >= 21.0)

This notebook loads the trained LightGBM models and the test set predictions to visualize and analyze model performance.

#### Sections:
1. **Overall Performance Metrics**: Detailed tabular summary and comparative bar plots.
2. **Feature Importance Analysis**: Understanding which tags, lags, or rolling statistics are most influential.
3. **Early Warning Visualization**: Plotting time-series trends of actual vs. predicted values during key alarm episodes in the test set.
4. **Precision-Recall Trade-off Analysis**: Evaluating how the classification threshold can be adjusted to balance false alarms and missed alarms.

--- 
## 1. Imports and Data Loading

In [ ]:
import os
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_curve, confusion_matrix, classification_report

# Setting plotting aesthetics
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 12
sns.set_style("whitegrid")
sns.set_palette("muted")

OUTPUT_DIR = r"outputs"
MODEL_DIR = r"models"

# Load test predictions
preds_path = os.path.join(OUTPUT_DIR, "test_predictions.parquet")
results_path = os.path.join(OUTPUT_DIR, "model_results_summary.csv")

if os.path.exists(preds_path) and os.path.exists(results_path):
    test_df = pd.read_parquet(preds_path)
    results_df = pd.read_csv(results_path, index_index=0 if 'index' in pd.read_csv(results_path).columns else None)
    print(f"Loaded test predictions shape: {test_df.shape}")
    print("Model results loaded successfully.")
else:
    print("Error: Model outputs not found. Make sure train_model.py has finished running.")

--- 
## 2. Compare Performance Across Horizons
Let's review the evaluation metrics (Precision, Recall, F1-Score, and False Alarm Rate) at our four horizons (5m, 15m, 30m, 60m).

In [ ]:
# Display summary table
summary = pd.read_csv(results_path, index_col=0)
display(summary[['precision', 'recall', 'f1', 'far', 'test_mae', 'test_rmse']])

# Bar plot comparing metrics
metrics_to_plot = ['precision', 'recall', 'f1', 'far']
plot_df = summary[metrics_to_plot].reset_index().rename(columns={'index': 'Horizon'})
plot_df['Horizon'] = plot_df['Horizon'].astype(str) + ' min'
plot_df_melted = pd.melt(plot_df, id_vars=['Horizon'], var_name='Metric', value_name='Score')

plt.figure(figsize=(12, 6))
ax = sns.barplot(x='Metric', y='Score', hue='Horizon', data=plot_df_melted)
plt.title("Alarm Prediction Performance by Horizon (Threshold = 21.0)", fontsize=14, pad=15)
plt.ylabel("Value (Scale 0-1)")
plt.ylim(0, 1.05)
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f'{height:.1%}', 
                    (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='center', 
                    xytext=(0, 8), 
                    textcoords='offset points', fontsize=10)
plt.tight_layout()
plt.show()

--- 
## 3. Feature Importance Analysis
Let's identify which sensors and engineered features have the most predictive power. We will examine the feature importances of the 15-minute and 30-minute models.

In [ ]:
# Load feature names
with open(os.path.join(OUTPUT_DIR, "feature_names.txt"), "r") as f:
    feature_cols = f.read().split("\n")

horizons = [5, 15, 30, 60]
for h in [5, 15, 30]:
    model_path = os.path.join(MODEL_DIR, f"lgb_model_{h}m.pkl")
    if os.path.exists(model_path):
        model = joblib.load(model_path)
        
        # Get feature importances
        importances = model.feature_importances_
        feat_imp = pd.DataFrame({'Feature': feature_cols, 'Importance': importances})
        feat_imp = feat_imp.sort_values(by='Importance', ascending=False).head(15)
        
        plt.figure(figsize=(12, 6))
        sns.barplot(x='Importance', y='Feature', data=feat_imp, palette='viridis')
        plt.title(f"Top 15 Most Important Features for {h}-Minute Horizon Model", fontsize=14)
        plt.xlabel("Split Importance (Number of times used in splits)")
        plt.tight_layout()
        plt.show()

--- 
## 4. Early Warning Time-Series Visualizations
Let's look at key alarm episodes in the test set. We plot the actual temperature (black line) and see how early our models predict the threshold crossing.

In [ ]:
# Find alarm episodes in the test set to plot
threshold = 21.0
test_df['in_alarm'] = test_df['actual'] >= threshold
test_df['alarm_change'] = test_df['in_alarm'].ne(test_df['in_alarm'].shift())
test_df['episode_id'] = test_df['alarm_change'].cumsum()

alarm_episodes = test_df[test_df['in_alarm']].groupby('episode_id').agg(
    start_time=('actual', lambda x: x.index.min()),
    end_time=('actual', lambda x: x.index.max()),
    duration_mins=('actual', lambda x: len(x)),
    max_val=('actual', 'max')
).reset_index(drop=True)

print(f"Found {len(alarm_episodes)} alarm episodes in the test set.")
print("Sample of episodes:")
display(alarm_episodes.sort_values(by='duration_mins', ascending=False).head(5))

# Plot representative episode (e.g. the longest one or a clear one)
if len(alarm_episodes) > 0:
    # Select the longest episode
    longest_ep = alarm_episodes.sort_values(by='duration_mins', ascending=False).iloc[0]
    start_plot = longest_ep['start_time'] - pd.Timedelta(hours=4)
    end_plot = longest_ep['end_time'] + pd.Timedelta(hours=4)
    
    plot_data = test_df.loc[start_plot:end_plot]
    
    plt.figure(figsize=(15, 8))
    plt.plot(plot_data.index, plot_data['actual'], label='Actual Temperature (03TIC_1023.PV)', color='black', linewidth=2.5)
    
    if 'pred_5m' in plot_data.columns:
        plt.plot(plot_data.index, plot_data['pred_5m'], label='5m Lead Prediction', color='purple', linestyle='--')
    if 'pred_15m' in plot_data.columns:
        # Plot shifted by its lead time so it lines up with when the actual alarm happened
        # i.e. we plot the 15m prediction at its target timestamp
        plt.plot(plot_data.index, plot_data['pred_15m'], label='15m Lead Prediction', color='green', linestyle='--')
    if 'pred_30m' in plot_data.columns:
        plt.plot(plot_data.index, plot_data['pred_30m'], label='30m Lead Prediction', color='blue', linestyle='--')
    if 'pred_60m' in plot_data.columns:
        plt.plot(plot_data.index, plot_data['pred_60m'], label='60m Lead Prediction', color='red', linestyle='--')
        
    plt.axhline(y=threshold, color='red', linestyle=':', label=f'Alarm Threshold ({threshold})', linewidth=2)
    plt.title("Alarm Episode Timeline: Actual Temperature vs. Multi-Horizon Predictions", fontsize=15, pad=15)
    plt.xlabel("Time")
    plt.ylabel("Value (degC)")
    plt.legend(loc='lower left', frameon=True, facecolor='white', framealpha=0.9)
    plt.tight_layout()
    plt.show()

--- 
## 5. Tuning Decision Thresholds (Precision-Recall Curves)
Depending on operational needs, we can optimize the threshold at which we flag a prediction as an alarm. A lower threshold will trigger alarms earlier (higher Recall) but may generate false alarms. A higher threshold will trigger fewer false alarms (higher Precision) but may miss some events.

In [ ]:
plt.figure(figsize=(10, 8))

for h, color in zip([5, 15, 30, 60], ['purple', 'green', 'blue', 'red']):
    pred_col = f'pred_{h}m'
    if pred_col in test_df.columns:
        # Clean data for precision-recall calculation
        clean_data = test_df[['actual', pred_col]].dropna()
        y_true_binary = (clean_data['actual'] >= threshold).astype(int)
        y_scores = clean_data[pred_col]
        
        precision, recall, thresholds_pr = precision_recall_curve(y_true_binary, y_scores)
        
        # Find the threshold closest to the default 21.0
        closest_idx = np.argmin(np.abs(thresholds_pr - 21.0))
        
        plt.plot(recall, precision, label=f'{h}m Horizon Model (AUC-PR)', color=color, linewidth=2)
        # Mark default 21.0 threshold
        plt.scatter(recall[closest_idx], precision[closest_idx], color=color, s=100, zorder=5,
                    edgecolors='black', label=f'{h}m default threshold (21.0)')

plt.xlabel('Recall (Sensitivity)', fontsize=12)
plt.ylabel('Precision (Positive Predictive Value)', fontsize=12)
plt.ylim([0.0, 1.05])
plt.xlim([0.0, 1.05])
plt.title('Precision-Recall Curves for Alarm Prediction', fontsize=14, pad=15)
plt.legend(loc="lower left")
plt.tight_layout()
plt.show()